In [1]:
from totalsegmentator.python_api import totalsegmentator
import SimpleITK as sitk, nibabel as nib
import numpy as np, os
from dicom2nifti.common import ConversionValidationError  # to catch that specific error

# ════════════════════════════════════════════════════════════════════════════════
# 📁 PARENT FOLDER CONTAINING MULTIPLE DICOM SUBFOLDERS
# ════════════════════════════════════════════════════════════════════════════════

parent_dir = r"D:\Abdomen_CT_Bone_Mets"
pelvis_labels = ["hip_left", "hip_right"]
lung_labels   = [
    "lung_upper_lobe_left",
    "lung_lower_lobe_left",
    "lung_upper_lobe_right",
    "lung_middle_lobe_right",
    "lung_lower_lobe_right"
]

# Sort subfolders by their trailing numeric part (e.g., BMAB1_00000001 → 1, BMAB1_00000002 → 2, etc.)
def sort_key(name):
    try:
        return int(name.split("_")[-1])
    except ValueError:
        return name  # fallback for nonmatching names

for subfolder in sorted(os.listdir(parent_dir), key=sort_key):
    dicom_dir = os.path.join(parent_dir, subfolder)
    if not os.path.isdir(dicom_dir):
        continue

    print(f"\n🔄 Processing folder: {dicom_dir}")

    # ─────────── Output folders and combined‐mask paths ───────────
    pelvis_out_dir = os.path.join(dicom_dir, "_seg_pelvis")
    lung_out_dir   = os.path.join(dicom_dir, "_seg_lung")

    pelvis_combined_f = os.path.join(pelvis_out_dir, "hips_combined.nii.gz")
    lung_combined_f   = os.path.join(lung_out_dir,   "lungs_combined.nii.gz")

    # If both combined outputs already exist, skip entire folder
    if os.path.exists(pelvis_combined_f) and os.path.exists(lung_combined_f):
        print(f"ℹ️  Skipping {subfolder}: combined outputs already exist.")
        continue

    # ─────── Helper: attempt segmentation on DICOM, else fallback to a SITK‐converted NIfTI ───────
    def run_totalseg_on(input_path, out_dir, labels):
        """
        Try totalsegmentator(input_path), and if a ConversionValidationError is raised,
        convert `input_path` (a DICOM folder) to a temporary NIfTI via SimpleITK, then re‐run totalseg.
        Returns nothing, but ensures that `out_dir/<label>.nii.gz` exists for each label.
        """
        try:
            totalsegmentator(
                input_path, out_dir,
                roi_subset=labels,
                fast=True,
                nr_thr_resamp=1, nr_thr_saving=1,
                device="gpu"
            )
        except ConversionValidationError:
            # 1) Convert DICOM folder → NIfTI using SITK
            print(f"⚠️  DICOM→NIfTI failed validation for {input_path}. Falling back to SimpleITK conversion.")
            reader = sitk.ImageSeriesReader()
            reader.SetFileNames(reader.GetGDCMSeriesFileNames(input_path))
            ct_itk = reader.Execute()
            # Write out a temporary NIfTI in the same folder
            nifti_tmp = os.path.join(input_path, "ct_converted.nii.gz")
            sitk.WriteImage(ct_itk, nifti_tmp)
            # 2) Run totalsegmentator on that NIfTI file
            totalsegmentator(
                nifti_tmp, out_dir,
                roi_subset=labels,
                fast=True,
                nr_thr_resamp=1, nr_thr_saving=1,
                device="gpu"
            )
            # Optionally remove the temporary NIfTI:
            os.remove(nifti_tmp)

    # ────────────── BLOCK 1A: Pelvis segmentation + combine hips ──────────────
    if not os.path.exists(pelvis_combined_f):
        os.makedirs(pelvis_out_dir, exist_ok=True)
        run_totalseg_on(dicom_dir, pelvis_out_dir, pelvis_labels)

        mask_combined = None
        for lab in pelvis_labels:
            fpath = os.path.join(pelvis_out_dir, f"{lab}.nii.gz")
            img   = nib.load(fpath)
            data  = img.get_fdata().astype(bool)

            if mask_combined is None:
                mask_combined = data
                affine, header = img.affine, img.header
            else:
                mask_combined |= data

        nib.save(nib.Nifti1Image(mask_combined.astype(np.uint8), affine, header), pelvis_combined_f)
        print("✅ Pelvis combined NIfTI saved ➜", pelvis_combined_f)
    else:
        print(f"ℹ️  {subfolder}: hips_combined.nii.gz already exists, skipping pelvis block.")

    # ────────────── BLOCK 1B: Lung segmentation + combine lobes ──────────────
    if not os.path.exists(lung_combined_f):
        os.makedirs(lung_out_dir, exist_ok=True)
        run_totalseg_on(dicom_dir, lung_out_dir, lung_labels)

        mask_combined = None
        for lab in lung_labels:
            fpath = os.path.join(lung_out_dir, f"{lab}.nii.gz")
            img   = nib.load(fpath)
            data  = img.get_fdata().astype(bool)

            if mask_combined is None:
                mask_combined = data
                affine, header = img.affine, img.header
            else:
                mask_combined |= data

        nib.save(nib.Nifti1Image(mask_combined.astype(np.uint8), affine, header), lung_combined_f)
        print("✅ Lung mask combined ➜", lung_combined_f)
    else:
        print(f"ℹ️  {subfolder}: lungs_combined.nii.gz already exists, skipping lung block.")


🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB1_00000001
ℹ️  Skipping BMAB1_00000001: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB2_00000001
ℹ️  Skipping BMAB2_00000001: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB3_00000001
ℹ️  Skipping BMAB3_00000001: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB1_00000002
ℹ️  Skipping BMAB1_00000002: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB2_00000002
ℹ️  Skipping BMAB2_00000002: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB3_00000002
ℹ️  Skipping BMAB3_00000002: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB1_00000003
ℹ️  Skipping BMAB1_00000003: combined outputs already exist.

🔄 Processing folder: D:\Abdomen_CT_Bone_Mets\BMAB2_00000003
ℹ️  Skipping BMAB2_00000003: combined outputs already exist.

🔄 Processing folder: D:

NotADirectoryError: [WinError 267] The directory name is invalid: 'C:\\Users\\RYANKR~1\\AppData\\Local\\Temp\\nnunet_tmp_arrzo4rw\\dcm\\converted_dcm.nii.gz'

In [ ]:
import nibabel as nib
import SimpleITK as sitk
import numpy as np
import pandas as pd
import os

parent_dir = r"D:\Abdomen_CT_Bone_Mets"
allowance_mm = 30.0

# reuse the same sorting logic as before so that BMAB1_00000001, BMAB1_00000002, … appear in order
def sort_key(name):
    try:
        return int(name.split("_")[-1])
    except ValueError:
        return name

rows = []

for subfolder in sorted(os.listdir(parent_dir), key=sort_key):
    dicom_dir = os.path.join(parent_dir, subfolder)
    if not os.path.isdir(dicom_dir):
        continue

    pelvis_mask_f = os.path.join(dicom_dir, "_seg_pelvis", "hips_combined.nii.gz")
    lung_mask_f   = os.path.join(dicom_dir, "_seg_lung",   "lungs_combined.nii.gz")

    # Skip if either combined mask is missing
    if not (os.path.exists(pelvis_mask_f) and os.path.exists(lung_mask_f)):
        print(f"⚠️  Skipping {subfolder}: missing mask files")
        continue

    # ——— Load CT & reorient to RAS ———
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(reader.GetGDCMSeriesFileNames(dicom_dir))
    ct_itk = reader.Execute()

    orienter = sitk.DICOMOrientImageFilter()
    orienter.SetDesiredCoordinateOrientation("RAS")
    ct_ras = orienter.Execute(ct_itk)
    z_spacing = ct_ras.GetSpacing()[-1]

    # ——— Diaphragm Detection (Cranial Overscan) ———
    mask_itk_lung = sitk.ReadImage(lung_mask_f)
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(ct_ras)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    mask_np_lung = sitk.GetArrayFromImage(resampler.Execute(mask_itk_lung)).astype(bool)

    z_indices_lung = np.where(mask_np_lung.any(axis=(1, 2)))[0]
    if z_indices_lung.size == 0:
        print(f"⚠️  {subfolder}: no lung voxels found → skipping")
        continue

    diaphragm_idx = int(z_indices_lung[-1])
    top_slice_index = mask_np_lung.shape[0] - 1
    distance_above_mm = (top_slice_index - diaphragm_idx) * z_spacing
    cranial_overscan_mm = max(0.0, distance_above_mm - allowance_mm)

    # ——— Pubic Symphysis Detection (Caudal Overscan) ———
    mask_np_pelvis = nib.load(pelvis_mask_f).get_fdata().astype(bool)
    z_dim, y_dim, x_dim = mask_np_pelvis.shape
    mid_x = x_dim // 2
    pubic_idx = None

    for z in range(z_dim):
        if mask_np_pelvis[z, :, max(0, mid_x - 2):mid_x + 3].any():
            pubic_idx = z
            break

    if pubic_idx is None:
        print(f"⚠️  {subfolder}: no symphysis found → skipping")
        continue

    distance_below_mm = pubic_idx * z_spacing
    caudal_overscan_mm = max(0.0, distance_below_mm - allowance_mm)

    rows.append({
        "subject": subfolder,
        "diaphragm_idx": diaphragm_idx,
        "distance_above_mm": distance_above_mm,
        "cranial_overscan_mm": cranial_overscan_mm,
        "pubic_idx": pubic_idx,
        "distance_below_mm": distance_below_mm,
        "caudal_overscan_mm": caudal_overscan_mm
    })

# Build DataFrame, save CSV, and print
df = pd.DataFrame(rows, columns=[
    "subject",
    "diaphragm_idx", "distance_above_mm", "cranial_overscan_mm",
    "pubic_idx", "distance_below_mm", "caudal_overscan_mm"
])

out_csv = os.path.join(parent_dir, "overscan_summary.csv")
df.to_csv(out_csv, index=False)

print("\n✅ Saved overscan_summary.csv ➜", out_csv)
print("\n=== DataFrame head ===")
print(df.head())
print("\n=== DataFrame summary (describe) ===")
print(df.describe())